# 01 — Exploration des données (chantier 2)

Prérequis (voir le README) :

```bash
uv run agridrone data download --include-candidates
uv run agridrone data prepare-classification
uv run agridrone data prepare-segmentation
```

Le notebook lit uniquement `data/processed/` (et les orthomosaïques de `data/interim/` pour la carte de
tuilage) et enregistre ses figures dans `reports/figures/`. Toute la logique de préparation est dans
`src/agridrone/data/` ; ici, on ne fait que regarder.

In [ ]:
import json

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Rectangle
from PIL import Image
from rasterio.enums import Resampling
from scipy import ndimage

from agridrone.core.paths import project_root

ROOT = project_root()
PROCESSED = ROOT / "data" / "processed"
INTERIM = ROOT / "data" / "interim"
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Palette de référence (validée : 3 premiers emplacements, toutes paires, mode clair).
SURFACE, INK, INK_2, MUTED, GRID, AXIS = (
    "#fcfcfb",
    "#0b0b0b",
    "#52514e",
    "#898781",
    "#e1e0d9",
    "#c3c2b7",
)
SLOT = ["#2a78d6", "#eb6834", "#1baf7a"]  # bleu, orange, aqua — ordre fixe
SPLIT_COLOR = {"train": SLOT[0], "val": SLOT[1], "test": SLOT[2]}
DIVERGING = LinearSegmentedColormap.from_list("rouge-gris-bleu", ["#e34948", "#f0efec", "#2a78d6"])

matplotlib.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "text.color": INK,
        "axes.labelcolor": INK_2,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "axes.edgecolor": AXIS,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "font.size": 9,
        "lines.linewidth": 2,
        "legend.frameon": False,
    }
)


def save(fig, name, dpi=150, fmt="png"):
    # JPEG pour les figures photographiques (bien plus légères dans le dépôt).
    path = FIG_DIR / f"{name}.{fmt}"
    kwargs = {"pil_kwargs": {"quality": 85}} if fmt == "jpg" else {}
    fig.savefig(path, dpi=dpi, bbox_inches="tight", **kwargs)
    print("figure :", path.relative_to(ROOT))

## 1. Classification des maladies — distribution des classes

Ensembles : `train` / `val` / `test` viennent de la source d'entraînement (John Innes), découpée par
groupes ; `test_<source>` sont les tests inter-sources (jamais vus à l'entraînement).

In [ ]:
manifest = pd.read_csv(PROCESSED / "classification" / "manifest.csv")
kept = manifest[manifest["excluded_reason"].isna() | (manifest["excluded_reason"] == "")]
CLASSES = ["healthy", "yellow_rust", "brown_rust", "septoria", "powdery_mildew"]
SPLITS = [
    "train",
    "val",
    "test",
    *sorted(s for s in kept["split"].unique() if s.startswith("test_")),
]
table = kept.pivot_table(
    index="label", columns="split", values="path", aggfunc="count", fill_value=0
)
table = table.reindex(index=CLASSES, columns=SPLITS, fill_value=0)
table.loc["total"] = table.sum()
table

In [ ]:
fig, axes = plt.subplots(1, len(SPLITS), figsize=(13, 2.8), sharey=True)
for ax, split in zip(axes, SPLITS, strict=True):
    counts = table.loc[CLASSES, split]
    ax.barh(CLASSES, counts, color=SLOT[0], height=0.6)
    ax.invert_yaxis()
    ax.set_title(split.replace("test_", "test · "), fontsize=9)
    ax.grid(axis="y", visible=False)
    for y, v in enumerate(counts):
        if v:
            ax.text(v, y, f" {v}", va="center", fontsize=8, color=INK_2)
    ax.set_xlim(0, counts.max() * 1.3 if counts.max() else 1)
    ax.tick_params(axis="x", labelsize=7)
fig.suptitle(
    "Images par classe et par ensemble (échelles indépendantes)",
    x=0.01,
    ha="left",
    fontweight="bold",
)
fig.tight_layout()
save(fig, "classif_distribution")

## 2. Exclusions et doublons

Chaque image écartée garde sa raison dans le manifeste (doublon exact, quasi-doublon d'une image
d'entraînement, quasi-doublon dans le même jeu de test…).

In [ ]:
excluded = manifest[manifest["excluded_reason"].fillna("") != ""]
excluded.groupby(["source", "excluded_reason"]).size().rename("images").to_frame()

## 3. Tailles d'image, appareils et dates de prise de vue

Le découpage « par date » n'est possible que si la date EXIF existe : on vérifie sa couverture.

In [ ]:
def describe(group):
    return pd.Series(
        {
            "images": len(group),
            "largeur médiane": int(group["width"].median()),
            "hauteur médiane": int(group["height"].median()),
            "plus petite": f"{group['width'].min()}×{group['height'].min()}",
            "plus grande": f"{group['width'].max()}×{group['height'].max()}",
            "avec date EXIF": int((group["exif_date"].fillna("") != "").sum()),
            "jours distincts": group.loc[
                group["exif_date"].fillna("") != "", "exif_date"
            ].nunique(),
            "appareils": ", ".join(sorted(set(group["camera"].dropna()) - {""})) or "—",
        }
    )


kept.groupby("source").apply(describe, include_groups=False)

## 4. Exemples visuels (quatre images par classe et par source)

In [ ]:
rng = np.random.default_rng(0)
sources = list(dict.fromkeys(kept["source"]))
fig, axes = plt.subplots(
    len(CLASSES), 4 * len(sources), figsize=(2.0 * 4 * len(sources), 2.0 * len(CLASSES))
)
for r, cls in enumerate(CLASSES):
    for s, source in enumerate(sources):
        pool = kept[(kept["label"] == cls) & (kept["source"] == source)]
        picks = pool.sample(min(4, len(pool)), random_state=0) if len(pool) else pool
        for k in range(4):
            ax = axes[r, 4 * s + k]
            ax.axis("off")
            if k < len(picks):
                with Image.open(INTERIM / picks.iloc[k]["path"]) as im:
                    im.draft("RGB", (160, 160))
                    ax.imshow(im.convert("RGB").resize((160, 160)))
            if r == 0 and k == 0:
                ax.set_title(source, fontsize=8, loc="left", color=INK_2)
            if s == 0 and k == 0:
                ax.text(
                    -0.08,
                    0.5,
                    cls,
                    transform=ax.transAxes,
                    rotation=90,
                    va="center",
                    ha="right",
                    fontsize=9,
                )
fig.tight_layout(w_pad=0.2, h_pad=0.2)
save(fig, "classif_exemples", dpi=80, fmt="jpg")

## 5. Multispectral — WeedsGalore (5 bandes, DJI P4 Multispectral, 2,5 mm/px)

Composite RVB, fausses couleurs (PIR, R, V), NDVI et masque. Les valeurs sont des comptes bruts
normalisés (pas de réflectance) : le NDVI est indicatif.

In [ ]:
wg_dir = PROCESSED / "segmentation" / "weedsgalore"
wg_info = json.loads((wg_dir / "dataset.json").read_text())
wg_index = pd.read_csv(wg_dir / "index.csv")
BANDS = wg_info["bands"]
b = {name: i for i, name in enumerate(BANDS)}
CLASS_COLOR = {
    "background": GRID,
    "crop": SLOT[0],
    "weed": SLOT[1],
}  # fond neutre, classes en ordre fixe


def stretch(rgb):
    lo, hi = np.percentile(rgb, [2, 98], axis=(0, 1))
    return np.clip((rgb - lo) / np.maximum(hi - lo, 1e-6), 0, 1)


def ndvi(image):
    nir, red = image[b["nir"]], image[b["red"]]
    return (nir - red) / (nir + red + 1e-6)


samples = (
    wg_index[wg_index["split"] == "train"]
    .sort_values("px_2", ascending=False)["sample"]
    .iloc[[0, 5]]
)
mask_cmap = ListedColormap([CLASS_COLOR[c] for c in wg_info["classes"]])
fig, axes = plt.subplots(len(samples), 4, figsize=(12, 3.4 * len(samples)), layout="constrained")
for row, sample in zip(axes, samples, strict=True):
    image = np.load(wg_dir / "images" / f"{sample}.npy")
    mask = np.asarray(Image.open(wg_dir / "masks" / f"{sample}.png"))
    row[0].imshow(stretch(image[[b["red"], b["green"], b["blue"]]].transpose(1, 2, 0)))
    row[1].imshow(stretch(image[[b["nir"], b["red"], b["green"]]].transpose(1, 2, 0)))
    im = row[2].imshow(ndvi(image), cmap=DIVERGING, vmin=-1, vmax=1)
    row[3].imshow(
        np.where(mask == 255, 0, mask),
        cmap=mask_cmap,
        vmin=0,
        vmax=len(wg_info["classes"]) - 1,
        interpolation="nearest",
    )
    for ax, title in zip(
        row, ["RVB", "fausses couleurs (PIR, R, V)", "NDVI", "masque"], strict=True
    ):
        ax.set_title(f"{title} — {sample}" if title == "RVB" else title, fontsize=9)
        ax.axis("off")
fig.colorbar(im, ax=axes[-1, 2], location="bottom", shrink=0.9, label="NDVI")
handles = [Rectangle((0, 0), 1, 1, color=CLASS_COLOR[c]) for c in wg_info["classes"]]
fig.legend(handles, wg_info["classes"], loc="outside lower right", ncol=3)
save(fig, "ms_weedsgalore_exemples", dpi=110, fmt="jpg")

## 6. Taille des objets en pixels

Composantes connexes des masques (plants de maïs, taches d'adventices). Avec un GSD de 2,5 mm/px,
1 px = 0,0625 cm². C'est le critère qui fixe la résolution nécessaire : un objet de quelques pixels
à 2,5 mm/px disparaît à 1 cm/px (vol plus haut).

In [ ]:
GSD_MM = 2.5
areas = {"crop": [], "weed": []}
for sample in wg_index["sample"]:
    mask = np.asarray(Image.open(wg_dir / "masks" / f"{sample}.png"))
    for name in areas:
        labeled, n = ndimage.label(mask == wg_info["classes"].index(name))
        if n:
            areas[name].extend(np.bincount(labeled.ravel())[1:].tolist())

rows = []
for name, values in areas.items():
    a = np.array(values)
    q = np.percentile(a, [10, 50, 90])
    rows.append(
        {
            "classe": name,
            "objets": len(a),
            "p10 (px)": int(q[0]),
            "médiane (px)": int(q[1]),
            "p90 (px)": int(q[2]),
            # À 1 cm/px, 1 px = 1 cm² : même valeur que la surface en cm².
            "médiane (cm² = px à 1 cm/px)": round(q[1] * (GSD_MM / 10) ** 2, 1),
        }
    )
sizes = pd.DataFrame(rows).set_index("classe")

fig, ax = plt.subplots(figsize=(7, 3))
bins = np.logspace(0, np.log10(max(max(v) for v in areas.values()) + 1), 40)
for name in areas:
    ax.hist(
        areas[name], bins=bins, histtype="step", color=CLASS_COLOR[name], linewidth=2, label=name
    )
ax.set_xscale("log")
ax.set_xlabel("surface d'un objet (pixels, échelle log)")
ax.set_ylabel("nombre d'objets")
ax.set_title("Taille des objets annotés (WeedsGalore, 2,5 mm/px)")
ax.legend()
save(fig, "ms_tailles_objets")
sizes

## 7. Séparabilité spectrale : NDVI par classe (ensemble d'entraînement)

Si une classe se sépare déjà par un seuil sur un indice, un réseau doit faire **mieux** que ce seuil pour
se justifier : c'est la comparaison prévue au chantier 5. Le seuil d'Otsu est calculé sur tous les
pixels d'entraînement.

In [ ]:
rng = np.random.default_rng(0)
pixels = {c: [] for c in wg_info["classes"]}
for sample in wg_index.loc[wg_index["split"] == "train", "sample"]:
    image = np.load(wg_dir / "images" / f"{sample}.npy")
    mask = np.asarray(Image.open(wg_dir / "masks" / f"{sample}.png"))
    nd = ndvi(image)
    for k, c in enumerate(wg_info["classes"]):
        values = nd[mask == k]
        if len(values) > 3000:
            values = rng.choice(values, 3000, replace=False)
        pixels[c].append(values)
pixels = {c: np.concatenate(v) for c, v in pixels.items()}


def otsu(values, bins=256):
    hist, edges = np.histogram(values, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(hist)
    w1 = w0[-1] - w0
    m0 = np.cumsum(hist * centers) / np.maximum(w0, 1)
    m1 = (np.sum(hist * centers) - np.cumsum(hist * centers)) / np.maximum(w1, 1)
    return centers[np.argmax(w0 * w1 * (m0 - m1) ** 2)]


threshold = otsu(np.concatenate(list(pixels.values())))
fig, ax = plt.subplots(figsize=(7, 3))
bins = np.linspace(-0.4, 1, 70)
for c, values in pixels.items():
    color = MUTED if c == "background" else CLASS_COLOR[c]
    ax.hist(values, bins=bins, density=True, histtype="step", color=color, linewidth=2, label=c)
ax.axvline(threshold, color=INK_2, linewidth=1, linestyle="--")
ax.text(
    threshold,
    ax.get_ylim()[1] * 0.95,
    f" Otsu = {threshold:.2f}",
    color=INK_2,
    fontsize=8,
    va="top",
)
ax.set_xlabel("NDVI (comptes bruts normalisés)")
ax.set_ylabel("densité")
ax.set_title("NDVI par classe — WeedsGalore, ensemble d'entraînement")
ax.legend()
save(fig, "ms_ndvi_par_classe")
pd.DataFrame(
    {
        c: {
            "pixels échantillonnés": len(v),
            "NDVI médian": round(float(np.median(v)), 3),
            "part > seuil d'Otsu": round(float((v > threshold).mean()), 3),
        }
        for c, v in pixels.items()
    }
).T

## 8. Maïs (Turquie) — patchs et pseudo-étiquettes

**Attention** : les masques sont dérivés d'indices de végétation par les auteurs. Ils servent à tester
la lecture des données, jamais à comparer un modèle à une baseline par indices.

In [ ]:
mz_dir = PROCESSED / "segmentation" / "maize_patches"
mz_info = json.loads((mz_dir / "dataset.json").read_text())
mz_index = pd.read_csv(mz_dir / "index.csv")
px = mz_index.groupby("split")[[f"px_{k}" for k in range(len(mz_info["classes"]))]].sum()
px.columns = mz_info["classes"]
(px.T / px.sum(axis=1)).round(3)

## 9. Tuilage d'une orthomosaïque et découpage spatial

Tuiles de 256 px avec 64 px de chevauchement, blocs de 1024 px tirés au sort entre train / val / test.
Les tuiles qui débordent sur un bloc d'un autre ensemble sont écartées : aucun pixel n'est partagé
entre deux ensembles.

In [ ]:
ortho_dir = INTERIM / "maize_stress_rust_tr" / "01_orthomosaics" / "common_rust_2025"
ortho = next(ortho_dir.glob("*.tif"))
tiles = pd.read_csv(
    PROCESSED / "segmentation" / "maize_orthotiles" / "common_rust_2025" / f"{ortho.stem}_tiles.csv"
)
factor = 8
with rasterio.open(ortho) as ds:
    shape = (ds.height // factor, ds.width // factor)
    rgb = ds.read([3, 2, 1], out_shape=(3, *shape), resampling=Resampling.average).astype("float32")
    alpha = ds.read(6, out_shape=shape, resampling=Resampling.nearest)
rgb = rgb.transpose(1, 2, 0)
valid = alpha > 0.5  # zones vides : alpha ~0 (pas de nodata dans cette orthomosaïque)
display = np.ones_like(rgb)
display[valid] = stretch(rgb[valid][:, None, :])[:, 0, :]

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(display)
for t in tiles.itertuples():
    ax.add_patch(
        Rectangle(
            (t.col / factor, t.row / factor),
            t.width / factor,
            t.height / factor,
            fill=False,
            edgecolor=SPLIT_COLOR[t.split],
            linewidth=0.8,
        )
    )
ax.axis("off")
handles = [
    Rectangle((0, 0), 1, 1, fill=False, edgecolor=c, linewidth=2) for c in SPLIT_COLOR.values()
]
counts = tiles["split"].value_counts()
ax.legend(
    handles,
    [f"{s} ({counts.get(s, 0)} tuiles)" for s in SPLIT_COLOR],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.01),
    ncol=3,
)
ax.set_title(f"Tuiles conservées par ensemble — {ortho.parent.name}")
save(fig, "tuilage_split_spatial", dpi=110, fmt="jpg")